# EcoScore Validation & Model Performance

In [ ]:
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from math import pi
sns.set_style('whitegrid')
print('Libraries imported')

## Load Data

In [ ]:
with open('../results/scores.json') as f: scores_data = json.load(f)
with open('../results/unet_metrics.json') as f: unet_metrics = json.load(f)
zone_scores = scores_data['zone_scores']
suppliers_df = pd.DataFrame(scores_data['suppliers'])
print(f'Loaded {len(suppliers_df)} suppliers and {len(unet_metrics["train_loss"])} epochs')

## 2. U-Net Training Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
epochs = np.arange(1, 11)
ax.plot(epochs, unet_metrics['train_loss'], 'o-', color='#3498DB', linewidth=2.5, label='Train Loss')
ax.plot(epochs, unet_metrics['test_loss'], 's-', color='#E67E22', linewidth=2.5, label='Test Loss')
ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax.set_ylabel('Loss', fontsize=12, fontweight='bold')
ax.set_title('U-Net Training Curve (Synthetic Data)', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)
ax.legend()
Path('../results/figures').mkdir(parents=True, exist_ok=True)
plt.savefig('../results/figures/07_unet_training_curve.png', dpi=300, bbox_inches='tight')
print('Saved training curve')
plt.show()

## 3. Model Comparison Table

In [ ]:
models = pd.DataFrame([
    {'Model': 'U-Net', 'Type': 'Segmentation', 'Metric': 'F1', 'Value': '1.0000', 'Data': 'Synthetic'},
    {'Model': 'U-Net', 'Type': 'Segmentation', 'Metric': 'IOU', 'Value': '1.0000', 'Data': 'Synthetic'},
    {'Model': 'EcoScore', 'Type': 'Composite', 'Metric': 'Suppliers', 'Value': '25', 'Data': 'Real'},
])
print('
' + '='*80)
print('MODEL COMPARISON')
print('='*80)
print(models.to_string(index=False))
print('='*80)

## 4. EcoScore Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
scores = suppliers_df['ecoscore'].values
ax.hist(scores, bins=15, color='#3498DB', edgecolor='black', alpha=0.7)
ax.axvline(40, color='#F39C12', linestyle='--', linewidth=2, label='REVIEW (40)')
ax.axvline(70, color='#2ECC71', linestyle='--', linewidth=2, label='COMPLIANT (70)')
ax.set_xlabel('EcoScore', fontsize=12, fontweight='bold')
ax.set_ylabel('Count', fontsize=12, fontweight='bold')
ax.set_title('EcoScore Distribution (40 Suppliers)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.savefig('../results/figures/08_ecoscore_distribution.png', dpi=300, bbox_inches='tight')
print('Saved distribution')
plt.show()

## 5. Zone Radar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
cats = ['Deforestation', 'Water', 'Pollution']
num = len(cats)
angles = [n/float(num)*2*np.pi for n in range(num)] + [0]
colors = {'pakistan': '#E74C3C', 'china': '#F39C12', 'bangladesh': '#3498DB'}
for zone, data in zone_scores.items():
    vals = [data['deforestation_risk'], data['water_stress'], data['pollution_proxy']] + [data['deforestation_risk']]
    ax.plot(angles, vals, 'o-', linewidth=2, label=zone.upper(), color=colors[zone])
    ax.fill(angles, vals, alpha=0.15, color=colors[zone])
ax.set_xticks(angles[:-1])
ax.set_xticklabels(cats, size=11, fontweight='bold')
ax.set_ylim(0, 100)
ax.grid(True)
ax.set_title('Zone Risk Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.savefig('../results/figures/09_zone_radar_chart.png', dpi=300, bbox_inches='tight')
print('Saved radar chart')
plt.show()

## 6. Summary

In [ ]:
print('
' + '='*80)
print('ECOSCORE VALIDATION SUMMARY')
print('='*80)
print(f'U-Net Test Loss: {unet_metrics["test_loss"][-1]:.4f}')
print(f'U-Net F1 Score: {unet_metrics["test_f1"][-1]:.4f}')
print(f'U-Net IOU: {unet_metrics["test_iou"][-1]:.4f}')
print(f'Total Suppliers: {len(suppliers_df)}')
print(f'Mean EcoScore: {suppliers_df["ecoscore"].mean():.2f}')
for status in ['COMPLIANT', 'REVIEW', 'CRITICAL']:
    cnt = len(suppliers_df[suppliers_df['status']==status])
    print(f'{status}: {cnt} ({cnt/len(suppliers_df)*100:.1f}%)')
print('='*80)